# ディープラーニングに入る前の橋渡し

ディープラーニングは、入力から予測を作り、損失を測り、パラメータを更新する流れを保ったまま、途中の表現も学習する方法です。機械学習で人が特徴量を作っていた部分を、層の重なりで作れるようにします。

小さな数値例で、線形モデル、ReLU 特徴量、2 層ネットワーク、損失、勾配更新、入力の形に合う層の選び方までを順に確認します。

## 線形モデルでは表しにくい形を見る

直線は入力が増えるほど出力も一定方向へ動く関係に向いています。V 字型のように途中で向きが変わる関係は、直線だけでは表しにくくなります。

In [ ]:
import numpy as np


In [ ]:
x = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=float)
y = np.abs(x)

linear_pred = np.full_like(y, fill_value=y.mean())
linear_mae = np.mean(np.abs(y - linear_pred))

print("x:", x)
print("target:", y)
print("linear prediction:", linear_pred)
print("linear MAE:", round(linear_mae, 4))


## ReLU で中間表現を作る

`ReLU(z) = max(0, z)` は、正の部分だけを残す変換です。`ReLU(x)` と `ReLU(-x)` を並べると、正方向と負方向を別々の特徴量として扱えます。

In [ ]:
def relu(values):
    return np.maximum(values, 0)

h1 = relu(x)
h2 = relu(-x)
features = np.column_stack([h1, h2])
feature_pred = h1 + h2
feature_mae = np.mean(np.abs(y - feature_pred))

print("features:")
print(features)
print("feature prediction:", feature_pred)
print("feature MAE:", round(feature_mae, 4))


## 層は入力を別の見方へ写す

ニューラルネットワークの層は、入力に重みを掛け、バイアスを足し、活性化関数を通して新しい表現を作ります。小さな2ユニットの隠れ層で、その形を確認します。

In [ ]:
X = x.reshape(-1, 1)
W1 = np.array([[1.0, -1.0]])
b1 = np.array([0.0, 0.0])
H = relu(X @ W1 + b1)
W2 = np.array([[1.0], [1.0]])
b2 = np.array([0.0])
network_pred = (H @ W2 + b2).ravel()

print("X shape:", X.shape)
print("H shape:", H.shape)
print("network prediction:", network_pred)


## 予測と損失を計算する

学習では、予測が正解からどれだけ外れたかを損失として測ります。平均二乗誤差は、大きな外れを強く罰する回帰用の損失です。

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

loss = mse(y, network_pred)
print("MSE:", round(loss, 6))
print("residual:", y - network_pred)


## パラメータを1回だけ更新する

勾配は、損失を下げるためにパラメータをどちらへ動かすかを示します。まずは1入力の線形モデルで、重みとバイアスを1回だけ更新します。

In [ ]:
x_one = 2.0
target = 1.0
w = 0.8
b = -0.3
lr = 0.1

pred = w * x_one + b
loss_before = (pred - target) ** 2
grad_w = 2 * (pred - target) * x_one
grad_b = 2 * (pred - target)

w_next = w - lr * grad_w
b_next = b - lr * grad_b
new_pred = w_next * x_one + b_next
loss_after = (new_pred - target) ** 2

print("pred before:", round(pred, 4), "loss:", round(loss_before, 4))
print("grad_w:", round(grad_w, 4), "grad_b:", round(grad_b, 4))
print("pred after:", round(new_pred, 4), "loss:", round(loss_after, 4))


## 複数データで少しずつ学習する

実際の学習では、全データの平均損失を使って更新を繰り返します。下の例では、直線モデルで V 字型を完全には表せませんが、更新によって損失が下がる様子を見ます。

In [ ]:
w = 0.0
b = 0.0
lr = 0.03

for step in range(6):
    pred = w * x + b
    loss = mse(y, pred)
    grad_w = np.mean(2 * (pred - y) * x)
    grad_b = np.mean(2 * (pred - y))
    print(step, "loss=", round(loss, 4), "w=", round(w, 4), "b=", round(b, 4))
    w -= lr * grad_w
    b -= lr * grad_b


## 表現を学ぶと形が増える

線形モデルだけでは足りないとき、中間層が新しい特徴量を作ります。深いモデルは、この特徴量作りを手作業ではなくデータから調整します。

In [ ]:
manual_features = np.column_stack([np.ones_like(x), relu(x), relu(-x)])
coef, *_ = np.linalg.lstsq(manual_features, y, rcond=None)
manual_pred = manual_features @ coef

print("coef:", np.round(coef, 4))
print("manual feature MAE:", round(np.mean(np.abs(y - manual_pred)), 6))
print("pred:", np.round(manual_pred, 3))


## 入力の形で層の設計が変わる

表、画像、系列、テキストでは、モデルに作ってほしい中間表現が違います。モデル名は、入力の形に合う見方を作りやすくする設計として読むと整理できます。

In [ ]:
design_choices = [
    {"input": "table", "structure": "列ごとの意味", "layer": "MLP"},
    {"input": "image", "structure": "近くの画素のまとまり", "layer": "CNN"},
    {"input": "sequence", "structure": "順番と過去の文脈", "layer": "RNN / Transformer"},
    {"input": "text", "structure": "離れた単語の関係", "layer": "Transformer"},
]

for row in design_choices:
    print(f"{row['input']:<8} -> {row['structure']} -> {row['layer']}")


## 学習で混ざりやすい役割を分ける

損失は学習で小さくする値、評価指標はモデルを比べる値、正則化は訓練データに寄りすぎないための制約です。役割を分けておくと、最適化と汎化を混同しにくくなります。

In [ ]:
terms = [
    ("parameter", "学習で更新される数"),
    ("activation", "中間表現を作る非線形変換"),
    ("loss", "学習中に小さくする値"),
    ("metric", "モデルを比較する値"),
    ("regularization", "訓練データへの寄りすぎを抑える工夫"),
]

for name, role in terms:
    print(f"{name:<15}: {role}")


ディープラーニングで増えるのは、入力から答えまでの間にある表現の層です。予測を作る、損失を測る、勾配で更新するという骨格は変わりません。この骨格を保ったまま、画像では近くの画素、系列では順番、テキストでは離れた関係を見るための層へ進みます。